That is why we wrote this Python script: It acts as a synthetic telemetric event logger that enriches static database orders with realistic, 
real-time API logs that standard e-commerce databases throw away.

In [1]:
import json
import random
import numpy as np
import pandas as pd
from sqlalchemy import create_engine

In [4]:
# 1. Database Connection
DB_USER = 'postgres'
DB_PASSWORD = 'Dikshu1998'  
DB_HOST = 'localhost'
DB_PORT = '5432'
DB_NAME = 'olist_db'

In [6]:
engine = create_engine(f'postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}')

In [7]:
print("Extracting data from postgreSQL")

query = """
SELECT 
    o.order_id,
    o.customer_id,
    op.payment_sequential,
    op.payment_type,
    op.payment_installments,
    op.payment_value,
    o.order_purchase_timestamp,
    o.order_approved_at
FROM olist_orders o
JOIN olist_order_payments op ON o.order_id = op.order_id
WHERE op.payment_type <> 'not_defined';
"""

df = pd.read_sql(query, con=engine)
print(f"fetched {len(df)}: records.")

Extracting data from postgreSQL
fetched 103883: records.


Multi-Gateway Routing Engine
Real-World Reality: Enterprise platforms never rely on a single payment processor. 
They practice Smart Payment Routing.Business Connection: 
The script assigns $60\%$ of traffic to Stripe (primary provider), $30\%$ to PayPal (secondary backup), and $10\%$ to Adyen (international processor). 
    This allows analysts downstream in Snowflake/Power BI to compare processor performance and negotiate lower interchange fees.

Payment gateways operate under strict Service Level Agreements (SLAs).
Business Connection:
A healthy API authorization should take around $350\text{ ms}$ ($0.35$ seconds).
The code sets a hardware floor of $110\text{ ms}$ because network packets cannot cross optical fiber cables faster than physics allows.
If your average latency jumps from $350\text{ ms}$ to $1,200\text{ ms}$, customer conversion drops by up to $7\%$.

When a payment fails, the engineering team must instantly classify whether it was a Business Failure or a System Failure.
Business Connection:   
HTTP 402 Payment Required (85% of failures): Soft declines caused by customer bank issues (insufficient_funds, expired_card).
Latency is normal (~$600\text{ ms}$). 
Action: Trigger an automated email asking the customer to update their card.
HTTP 504 Gateway Timeout (15% of failures): Hard server crashes where Stripe/PayPal fails to answer within $4,500\text{ ms}$ ($4.5$ seconds). 
Action: Trigger an automated PagerDuty alarm for DevOps engineers to reroute traffic to a backup gateway.

In [13]:
# 2. Synthetic Gateway Log Generator
gateways = ['Stripe', 'PayPal', 'Adyen']
gateway_weights = [0.60, 0.30, 0.10]

decline_reasons = [
    'insufficient_funds',
    'card_declined',
    'suspected_fraud',
    'expired_card',
]


def generate_gateway_event(row):
    provider = str(np.random.choice(gateways, p=gateway_weights))
    is_approved = pd.notnull(row['order_approved_at'])

    if is_approved:
        status_code = 200
        decline_reason = None
        latency_ms = int(max(110, np.random.normal(350, 100)))
    else:
        status_code = int(np.random.choice([402, 504], p=[0.85, 0.15]))
        decline_reason = (
            str(random.choice(decline_reasons))
            if status_code == 402
            else 'gateway_timeout'
        )
        latency_ms = (
            int(np.random.normal(4500, 500))
            if status_code == 504
            else int(np.random.normal(600, 150))
        )

    # Nest payload as JSON structure
    log_payload = {
        'transaction_id': f'txn_{str(row["order_id"])[:12]}_{int(row["payment_sequential"])}',
        'order_id': str(row['order_id']),
        'gateway_provider': provider,
        'payment_method': str(row['payment_type']),
        'amount': float(row['payment_value']),
        'installments': int(row['payment_installments']),
        'response': {
            'status_code': status_code,
            'latency_ms': latency_ms,
            'decline_reason': decline_reason,
            'is_flagged_fraud': True if decline_reason == 'suspected_fraud' else False,
        },
        'created_at': str(row['order_purchase_timestamp']),
    }
    return json.dumps(log_payload)


print("2. Generating synthetic JSON gateway logs...")
df['gateway_log_json'] = df.apply(generate_gateway_event, axis=1)

# 3. Save JSON payload file
output_file = 'olist_stripe_paypal_gateway_logs.json'
print(f"3. Writing logs to '{output_file}'...")

with open(output_file, 'w') as f:
    for log in df['gateway_log_json']:
        f.write(log + '\n')

print(f" Success! Generated JSON log file for {len(df):,} transactions.")

2. Generating synthetic JSON gateway logs...
3. Writing logs to 'olist_stripe_paypal_gateway_logs.json'...
 Success! Generated JSON log file for 103,883 transactions.


APIs do not speak in flat SQL tables—they send nested JSON payloads
Business Connection:
By generating nested JSON strings, we mirror the exact structure of a Stripe Webhook API Payload.
This prepares our architecture for Snowflake, 
where we will use Snowflake's native VARIANT data type to parse and flatten semi-structured JSON logs at scale using SQL.

In [14]:
# 1. Print first 3 raw JSON log lines
print("--- RAW JSON LINES ---")
with open('olist_stripe_paypal_gateway_logs.json', 'r') as f:
    for i in range(3):
        print(f.readline().strip())

# 2. Load into Pandas as JSON Lines DataFrame
df_logs = pd.read_json('olist_stripe_paypal_gateway_logs.json', lines=True)
print("\n--- PARSED DATAFRAME ---")
print(df_logs.head())

--- RAW JSON LINES ---
{"transaction_id": "txn_b81ef226f3fe_1", "order_id": "b81ef226f3fe1789b1e8b2acac839d17", "gateway_provider": "Stripe", "payment_method": "credit_card", "amount": 99.33, "installments": 8, "response": {"status_code": 200, "latency_ms": 348, "decline_reason": null, "is_flagged_fraud": false}, "created_at": "2018-04-25 22:01:49"}
{"transaction_id": "txn_a9810da82917_1", "order_id": "a9810da82917af2d9aefd1278f1dcfa0", "gateway_provider": "Stripe", "payment_method": "credit_card", "amount": 24.39, "installments": 1, "response": {"status_code": 200, "latency_ms": 381, "decline_reason": null, "is_flagged_fraud": false}, "created_at": "2018-06-26 11:01:38"}
{"transaction_id": "txn_3d7239c394a2_1", "order_id": "3d7239c394a212faae122962df514ac7", "gateway_provider": "Stripe", "payment_method": "credit_card", "amount": 51.84, "installments": 3, "response": {"status_code": 200, "latency_ms": 472, "decline_reason": null, "is_flagged_fraud": false}, "created_at": "2017-06-05 1